# VLS workflow (Kaiyu)

This notebook is a clean version of the VLS setup workflow for config 2.

Process:
1. Set all configuration values at the top (run numbers, mode, ROI).
2. Load dark data and build a background spectrum.
3. Load x-ray data and subtract the background.
4. Compute per-shot VLS moments and compare VLS sum with GMD.
5. Compute the global XAS-style normalization ratio $\sum GMD / \sum VLS$.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Resolve repo root robustly (works from notebook dir or repo root).
cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd] + list(cwd.parents):
    if (p / "analysis" / "scripts").exists():
        repo_root = p
        break
if repo_root is None:
    raise RuntimeError("Could not locate repository root containing analysis/scripts")

sys.path.insert(0, str(repo_root / "analysis" / "scripts"))

import config
from data_loading import load_data, load_raw_h5

%matplotlib inline

## 1) Configuration (edit this first)

Change run numbers and file paths here.

In [ ]:
# Data source mode
USE_RAW_H5 = True
MAX_FILES = 5

# Run numbers (used when USE_RAW_H5 = True)
DARK_RUN_NO = 58764
XRAY_RUN_NO = 58766

# Combined H5 files (used when USE_RAW_H5 = False)
DARK_H5_FILE = config.COMBINED_DIR / "test_config2.h5"
XRAY_H5_FILE = config.COMBINED_DIR / "test_config2.h5"
TRIM_START = 3
TRIM_END = 3
DOWNSAMPLE_N = 1

# Shared preprocessing
ROI = (500, 600)  # Set to None for full pixel range

print(f"USE_RAW_H5: {USE_RAW_H5}")
if USE_RAW_H5:
    print(f"Dark run: {DARK_RUN_NO}, X-ray run: {XRAY_RUN_NO}, max_files: {MAX_FILES}")
else:
    print(f"Dark file: {DARK_H5_FILE}")
    print(f"X-ray file: {XRAY_H5_FILE}")
print(f"ROI: {ROI}")

## 2) Helper loader

In [ ]:
def load_cfg2_dataset(*, run_no=None, h5_path=None, roi=None):
    if USE_RAW_H5:
        if run_no is None:
            raise ValueError("run_no is required in raw mode")
        data = load_raw_h5(run_no, config=2, max_files=MAX_FILES)
        label = f"raw run {run_no}"
    else:
        if h5_path is None:
            raise ValueError("h5_path is required in combined-H5 mode")
        data = load_data(
            str(h5_path),
            config=2,
            trim_start=TRIM_START,
            trim_end=TRIM_END,
            downsample_N=DOWNSAMPLE_N,
        )
        label = str(h5_path)

    if roi is not None:
        data = data.crop_vls(*roi)

    return data, label

## 3) Load dark data and build background

Background is the mean VLS spectrum per bunch index from the dark run.

In [ ]:
dark_data, dark_label = load_cfg2_dataset(
    run_no=DARK_RUN_NO,
    h5_path=DARK_H5_FILE,
    roi=ROI,
)

print(f"Dark source: {dark_label}")
print(f"Dark VLS shape: {dark_data.vls.shape}")

mean_by_bunch_dark = np.nanmean(dark_data.vls, axis=0)
bg = mean_by_bunch_dark.copy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(dark_data.vls_pixels, np.nanmean(mean_by_bunch_dark, axis=0), color="steelblue")
axes[0].set_title("Dark mean spectrum (averaged over bunches)")
axes[0].set_xlabel("Pixel")
axes[0].set_ylabel("Intensity (arb.)")

axes[1].plot(np.nansum(mean_by_bunch_dark, axis=1), np.arange(mean_by_bunch_dark.shape[0]), marker="o", ms=2, ls="")
axes[1].set_title("Dark integrated intensity per bunch")
axes[1].set_xlabel("Sum intensity")
axes[1].set_ylabel("Bunch index")

fig.tight_layout()
plt.show()

## 4) Load x-ray data and subtract background

In [ ]:
xray_data, xray_label = load_cfg2_dataset(
    run_no=XRAY_RUN_NO,
    h5_path=XRAY_H5_FILE,
    roi=ROI,
)

print(f"X-ray source: {xray_label}")
print(f"X-ray VLS shape before subtraction: {xray_data.vls.shape}")

if xray_data.vls.shape[1:] != bg.shape:
    raise ValueError(
        f"Shape mismatch: xray per-train shape {xray_data.vls.shape[1:]} vs bg shape {bg.shape}"
    )

data = xray_data.subtract_background(bg)
print(f"VLS shape after subtraction: {data.vls.shape}")
print(f"Value range after subtraction: {np.nanmin(data.vls):.2f} .. {np.nanmax(data.vls):.2f}")

## 5) Quick inspection after subtraction

In [ ]:
pixel_ax = data.vls_pixels
flat = data.vls.reshape(-1, data.vls.shape[-1])
mean_spec = np.nanmean(flat, axis=0)
std_spec = np.nanstd(flat, axis=0)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(pixel_ax, mean_spec, color="mediumseagreen", lw=1.2, label="mean spectrum")
ax.fill_between(pixel_ax, mean_spec - std_spec, mean_spec + std_spec, color="mediumseagreen", alpha=0.25, label="+-1 std")
ax.set_xlabel("Pixel")
ax.set_ylabel("Intensity (arb.)")
ax.set_title("Background-subtracted VLS average")
ax.legend()
fig.tight_layout()
plt.show()

## 6) Compute moments, correlate VLS sum with GMD, and compute XAS ratio

This follows docs/how_to_do_XAS.md:
- check VLS sum vs GMD correlation
- compute global normalization ratio $\sum GMD / \sum VLS$

In [ ]:
data = data.compute_vls_moments()

gmd_flat = data.gmd.ravel()
vls_sum_flat = data.vls_sums.ravel()
good = np.isfinite(gmd_flat) & np.isfinite(vls_sum_flat)
x = gmd_flat[good]
y = vls_sum_flat[good]

corr = float(np.corrcoef(x, y)[0, 1])
sum_gmd = float(np.sum(x))
sum_vls = float(np.sum(y))
xas_ratio = sum_gmd / sum_vls

print(f"Valid shots: {x.size}")
print(f"Pearson r(GMD, VLS sum): {corr:.4f}")
print(f"sum(GMD): {sum_gmd:.6g}")
print(f"sum(VLS sum): {sum_vls:.6g}")
print(f"XAS normalization ratio sum(GMD)/sum(VLS): {xas_ratio:.6g}")

fig, ax = plt.subplots(figsize=(6, 5))
bins_x = np.linspace(np.percentile(x, 0.5), np.percentile(x, 99.5), 60)
bins_y = np.linspace(np.percentile(y, 0.5), np.percentile(y, 99.5), 60)
h = ax.hist2d(x, y, bins=[bins_x, bins_y], cmap="viridis", cmin=1, norm=mcolors.LogNorm())
fig.colorbar(h[3], ax=ax, label="Shots per bin")
ax.set_xlabel("GMD (uJ)")
ax.set_ylabel("VLS sum (arb.)")
ax.set_title("GMD vs VLS sum")
fig.tight_layout()
plt.show()

## 7) Optional: bin by photon energy (MPE)

If you want energy-resolved behavior, bin shots by MPE and inspect
the average VLS sum in each bin.

In [ ]:
mpe_flat = data.mpe_broadcast.ravel()
good_mpe = good & np.isfinite(mpe_flat)
mpe = mpe_flat[good_mpe]
vls_for_mpe = vls_sum_flat[good_mpe]

n_bins = 12
edges = np.linspace(np.percentile(mpe, 1), np.percentile(mpe, 99), n_bins + 1)
idx = np.digitize(mpe, edges) - 1

centers = 0.5 * (edges[:-1] + edges[1:])
mean_vls = np.full(n_bins, np.nan)
for i in range(n_bins):
    in_bin = idx == i
    if np.any(in_bin):
        mean_vls[i] = np.nanmean(vls_for_mpe[in_bin])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(centers, mean_vls, marker="o")
ax.set_xlabel("MPE (eV)")
ax.set_ylabel("Mean VLS sum (arb.)")
ax.set_title("Binned by central photon energy (MPE)")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()